# Decision Rules: Cheap Methods vs DIIM

**Question:** When can a cheap structural method substitute for the expensive DIIM simulation to identify the key economic sectors?

**Cheap methods** (fixed ranking, no simulation needed):
1. PCA (multi-PC distance on H = A * L)
2. Network Centrality (eigenvector + PageRank + betweenness)
3. Backward Linkage (column sums of Leontief inverse)
4. Forward Linkage (row sums of Leontief inverse)
5. Output-Weighted Linkage

**Approach:**
- For each of 2 scenarios (COVID, Manpower), run 2000 Monte Carlo trials with random q0.
- In each trial, compare each cheap method's loss reduction against DIIM's.
- If a cheap method achieves >= 95% of DIIM's reduction, it is "close".
- Fit logistic regression to find which q0 features predict when a cheap method will be close.

**Note:** Runtime ~30+ minutes (2000 MC x 2 scenarios x 5 methods).

## 1. Load Libraries & Functions

In [ ]:
setwd("..")

library(openxlsx)
library(igraph)
library(ggplot2)
library(reshape2)
library(dplyr)

source_notebook <- function(nb_path) {
  nb <- jsonlite::fromJSON(nb_path, simplifyVector = FALSE)
  code_cells <- Filter(function(cell) cell$cell_type == "code", nb$cells)
  code_lines <- unlist(lapply(code_cells, function(cell) {
    paste(unlist(cell$source), collapse = "")
  }))
  all_code <- paste(code_lines, collapse = "\n")
  eval(parse(text = all_code), envir = globalenv())
}

source_notebook("functions.ipynb")

results_dir <- "results"
dir.create(results_dir, showWarnings = FALSE, recursive = TRUE)

## 2. Column-Name Lookup

Throughout this notebook, each cheap method needs a short prefix for column names (e.g., `PCA_share`, `BackwardLinkage_ratio`). We define a lookup table so we don't have to use regex everywhere.

In [ ]:
# full method names (used for display and as keys)
method_names <- c("PCA", "Network Centrality", "Backward Linkage",
                  "Forward Linkage", "Output-Weighted Linkage")

# short column prefixes (letters only, no spaces/hyphens)
method_prefixes <- c(
    "PCA"                     = "PCA",
    "Network Centrality"      = "NetworkCentrality",
    "Backward Linkage"        = "BackwardLinkage",
    "Forward Linkage"         = "ForwardLinkage",
    "Output-Weighted Linkage" = "OutputWeightedLinkage"
)

# short labels for plots
method_labels <- c("PCA", "Network Centrality", "Backward Linkage",
                   "Forward Linkage", "Output-Weighted Linkage")

## 3. Compute q0 Features

Given a random `q0` vector and the cheap rankings, compute features that summarise how well each method's top-k sectors capture the disruption. These features are used later as predictors in logistic regression.

For each method we compute:
- `{prefix}_share`: fraction of total q0 captured by the method's top-k sectors
- `{prefix}_avgrank`: average rank of the method's sectors (normalised, lower = better)
- `{prefix}_overlap`: fraction of method's top-k that overlap with q0's own top-k

Plus 4 general q0 features: Gini, CV, max ratio, normalised entropy.

In [ ]:
compute_q0_features <- function(q0, cheap_rankings, k = 5) {
    n <- length(q0)
    total_q0 <- sum(q0)
    features <- list()

    for (method_name in names(cheap_rankings)) {
        top_k  <- cheap_rankings[[method_name]][1:k]
        prefix <- method_prefixes[method_name]

        # share of total q0 captured by this method's top-k
        features[[paste0(prefix, "_share")]] <- sum(q0[top_k]) / total_q0

        # average rank of method's sectors in q0 ordering (normalised 0-1)
        q0_ranks <- rank(-q0)
        features[[paste0(prefix, "_avgrank")]] <- mean(q0_ranks[top_k]) / n

        # overlap: how many of top-k by q0 are in method's top-k
        q0_topk <- order(q0, decreasing = TRUE)[1:k]
        features[[paste0(prefix, "_overlap")]] <- length(intersect(top_k, q0_topk)) / k
    }

    # general q0 distribution features
    features[["q0_gini"]] <- 1 - 2 * sum((1:n) * sort(q0)) / (n * total_q0) + (n + 1) / n
    features[["q0_cv"]]   <- sd(q0) / mean(q0)
    features[["q0_max_ratio"]] <- max(q0) / mean(q0)
    features[["q0_entropy"]] <- {
        p <- q0 / total_q0
        p <- p[p > 0]
        -sum(p * log(p)) / log(n)   # normalised entropy
    }

    return(as.data.frame(features))
}

## 4. Monte Carlo Runner

For a given scenario (COVID or Manpower):
1. Load data, compute cheap rankings once.
2. For each of 2000 trials, draw a random `q0`, run DIIM baseline, then DIIM with each method's top-k sectors.
3. Record reduction ratios and whether each cheap method is "close" (>= 95% of DIIM) or "wins" (>= DIIM).

In [ ]:
run_monte_carlo <- function(scenario_name, data_loader,
                            lockdown_duration = 55, total_duration = 751,
                            days_in_year = 366, n_mc = 2000, k = 5) {
    cat(sprintf("\n%s: %d Monte Carlo trials (k=%d)\n", scenario_name, n_mc, k))

    data <- data_loader()
    A      <- data$A
    x      <- data$x
    q0_base <- data$q0
    q0_base[q0_base == 0] <- 1e-8
    c_star <- data$c_star
    A_star <- data$A_star
    n      <- length(q0_base)
    max_q0 <- max(q0_base) * 2

    # compute cheap rankings once
    cat("  Computing cheap rankings...\n")
    cheap_rankings <- compute_cheap_rankings(A, A_star, x)
    for (m in names(cheap_rankings)) {
        cat(sprintf("    %-25s top-%d: [%s]\n", m, k,
            paste(cheap_rankings[[m]][1:k], collapse = ", ")))
    }

    # monte carlo loop
    cat("  Running Monte Carlo...\n")
    set.seed(42)
    all_results <- list()

    for (trial in 1:n_mc) {
        q0_rand <- runif(n, min = 1e-6, max = max_q0)

        # DIIM baseline (no key sectors)
        base_model <- DIIM(q0_rand, A_star, c_star, x,
            lockdown_duration, total_duration, days_in_year = days_in_year)
        base_loss <- base_model$total_economic_loss
        if (base_loss < 1e-6) next

        # DIIM with its own top-k (the gold standard)
        max_el <- apply(base_model$EL_evolution, 1, max)
        diim_topk <- order(max_el, decreasing = TRUE)[1:k]
        diim_model <- DIIM(q0_rand, A_star, c_star, x,
            lockdown_duration, total_duration,
            key_sectors = diim_topk, days_in_year = days_in_year)
        diim_reduction <- base_loss - diim_model$total_economic_loss
        if (diim_reduction < 1e-10) next

        # start building the row with q0 features
        trial_row <- compute_q0_features(q0_rand, cheap_rankings, k)
        trial_row$trial     <- trial
        trial_row$scenario  <- scenario_name
        trial_row$base_loss <- base_loss
        trial_row$diim_reduction <- diim_reduction
        trial_row$diim_pct  <- diim_reduction / base_loss * 100

        # run each cheap method
        for (m in names(cheap_rankings)) {
            topk <- cheap_rankings[[m]][1:k]
            m_model <- DIIM(q0_rand, A_star, c_star, x,
                lockdown_duration, total_duration,
                key_sectors = topk, days_in_year = days_in_year)
            m_reduction <- base_loss - m_model$total_economic_loss

            prefix <- method_prefixes[m]
            trial_row[[paste0(prefix, "_reduction")]] <- m_reduction
            trial_row[[paste0(prefix, "_ratio")]]     <- m_reduction / diim_reduction
            trial_row[[paste0(prefix, "_wins")]]      <- m_reduction >= diim_reduction
            trial_row[[paste0(prefix, "_close")]]     <- m_reduction / diim_reduction >= 0.95
        }

        all_results[[length(all_results) + 1]] <- trial_row

        if (trial %% 200 == 0) {
            pca_close_rate <- mean(sapply(all_results, function(r) r$PCA_close), na.rm = TRUE)
            cat(sprintf("    %d/%d done (PCA close rate: %.1f%%)\n",
                trial, n_mc, pca_close_rate * 100))
        }
    }

    mc_df <- bind_rows(all_results)
    cat(sprintf("  Valid trials: %d\n", nrow(mc_df)))
    return(list(mc_df = mc_df, cheap_rankings = cheap_rankings))
}

## 5. Build Decision Rules (Logistic Regression)

For each cheap method, fit a logistic regression predicting "close" (>= 95% of DIIM) from q0 features. Then find the best single predictor and sweep thresholds to get the optimal decision rule using Youden's J statistic.

In [ ]:
build_decision_rules <- function(mc_df, scenario_name, methods) {
    cat(sprintf("\nDecision Rules for %s\n", scenario_name))
    rules <- list()

    for (m in methods) {
        prefix      <- method_prefixes[m]
        close_col   <- paste0(prefix, "_close")
        ratio_col   <- paste0(prefix, "_ratio")
        share_col   <- paste0(prefix, "_share")
        avgrank_col <- paste0(prefix, "_avgrank")
        overlap_col <- paste0(prefix, "_overlap")

        if (!close_col %in% names(mc_df)) next

        close_rate <- mean(mc_df[[close_col]], na.rm = TRUE)
        win_rate   <- mean(mc_df[[paste0(prefix, "_wins")]], na.rm = TRUE)
        mean_ratio <- mean(mc_df[[ratio_col]], na.rm = TRUE)

        cat(sprintf("\n  %-25s close=%.1f%% wins=%.1f%% ratio=%.3f\n",
            m, close_rate * 100, win_rate * 100, mean_ratio))

        # feature columns for logistic regression
        feature_cols <- c(share_col, avgrank_col, overlap_col,
                          "q0_gini", "q0_cv", "q0_max_ratio", "q0_entropy")
        feature_cols <- feature_cols[feature_cols %in% names(mc_df)]

        if (length(feature_cols) == 0 || close_rate == 0 || close_rate == 1) {
            cat("    Skipping logistic regression (degenerate)\n")
            rules[[m]] <- list(method = m, close_rate = close_rate,
                win_rate = win_rate, mean_ratio = mean_ratio,
                model = NULL, threshold = NA, best_predictor = NA)
            next
        }

        formula_str <- paste(close_col, "~", paste(feature_cols, collapse = " + "))
        tryCatch({
            lr_model <- glm(as.formula(formula_str), data = mc_df, family = binomial)

            # find the most significant predictor (lowest p-value)
            coeffs <- summary(lr_model)$coefficients
            if (nrow(coeffs) > 1) {
                p_vals    <- coeffs[2:nrow(coeffs), 4]
                best_pred <- names(which.min(p_vals))
            } else {
                best_pred <- NA
            }

            cat(sprintf("    Best predictor: %s (p=%.4f)\n",
                best_pred, min(coeffs[2:nrow(coeffs), 4])))

            # sweep thresholds on the best predictor to find Youden's J optimal
            if (!is.na(best_pred) && best_pred %in% names(mc_df)) {
                pred_vals  <- mc_df[[best_pred]]
                close_vals <- mc_df[[close_col]]
                thresholds <- quantile(pred_vals, probs = seq(0.05, 0.95, 0.05))
                best_j     <- -Inf
                best_thresh <- NA

                for (thresh in thresholds) {
                    if (grepl("avgrank", best_pred)) {
                        predicted <- pred_vals < thresh  # lower rank = better
                    } else {
                        predicted <- pred_vals > thresh
                    }
                    tp <- sum(predicted & close_vals)
                    tn <- sum(!predicted & !close_vals)
                    fp <- sum(predicted & !close_vals)
                    fn <- sum(!predicted & close_vals)

                    sens <- tp / max(tp + fn, 1)
                    spec <- tn / max(tn + fp, 1)
                    j    <- sens + spec - 1

                    if (j > best_j) {
                        best_j      <- j
                        best_thresh <- thresh
                    }
                }

                direction <- if (grepl("avgrank", best_pred)) "<" else ">"
                cat(sprintf("    Rule: Use %s if %s %s %.3f (Youden J=%.3f)\n",
                    m, best_pred, direction, best_thresh, best_j))

                rules[[m]] <- list(method = m, close_rate = close_rate,
                    win_rate = win_rate, mean_ratio = mean_ratio,
                    model = lr_model, best_predictor = best_pred,
                    threshold = best_thresh, direction = direction,
                    youden_j = best_j)
            } else {
                rules[[m]] <- list(method = m, close_rate = close_rate,
                    win_rate = win_rate, mean_ratio = mean_ratio,
                    model = lr_model, best_predictor = best_pred,
                    threshold = NA)
            }
        }, error = function(e) {
            cat(sprintf("    Logistic regression failed: %s\n", e$message))
            rules[[m]] <<- list(method = m, close_rate = close_rate,
                win_rate = win_rate, mean_ratio = mean_ratio, model = NULL)
        })
    }

    return(rules)
}

## 6. Plot Generator

Three types of plots per scenario:
1. **Ratio distributions** — density plot of each method's performance ratio.
2. **Close rates** — horizontal bar chart showing how often each method is >= 95% of DIIM.
3. **Decision boundary** — scatter plot for each method with a rule, showing the threshold.

In [ ]:
generate_plots <- function(mc_df, rules, scenario_name, prefix) {
    clean_methods <- c("PCA", "NetworkCentrality", "BackwardLinkage",
                       "ForwardLinkage", "OutputWeightedLinkage")

    # --- Plot 1: Performance ratio distributions ---
    ratio_data <- data.frame()
    for (i in seq_along(clean_methods)) {
        col <- paste0(clean_methods[i], "_ratio")
        if (col %in% names(mc_df)) {
            ratio_data <- rbind(ratio_data, data.frame(
                method = method_labels[i],
                ratio  = mc_df[[col]],
                stringsAsFactors = FALSE))
        }
    }

    if (nrow(ratio_data) > 0) {
        p1 <- ggplot(ratio_data, aes(x = ratio, fill = method)) +
            geom_density(alpha = 0.5) +
            geom_vline(xintercept = 0.95, linetype = "dashed", color = "red", linewidth = 0.8) +
            geom_vline(xintercept = 1.0, linetype = "dotted", color = "darkgreen", linewidth = 0.8) +
            annotate("text", x = 0.95, y = Inf, label = "95% threshold",
                     vjust = 2, hjust = 1.1, color = "red", size = 3) +
            scale_fill_brewer(palette = "Set2") +
            labs(title = sprintf("%s: Performance Ratio Distributions", scenario_name),
                 subtitle = "Ratio = cheap method reduction / DIIM reduction",
                 x = "Performance Ratio", y = "Density", fill = "Method") +
            theme_minimal() +
            theme(plot.title = element_text(face = "bold", size = 14),
                  legend.position = "bottom")
        ggsave(file.path(results_dir, paste0(prefix, "_ratio_distributions.png")),
               p1, width = 12, height = 7)
        cat(sprintf("  Saved %s_ratio_distributions.png\n", prefix))
    }

    # --- Plot 2: Close rates bar chart ---
    close_data <- data.frame()
    for (i in seq_along(clean_methods)) {
        col <- paste0(clean_methods[i], "_close")
        if (col %in% names(mc_df)) {
            close_data <- rbind(close_data, data.frame(
                method     = method_labels[i],
                close_rate = mean(mc_df[[col]], na.rm = TRUE),
                win_rate   = mean(mc_df[[paste0(clean_methods[i], "_wins")]], na.rm = TRUE),
                stringsAsFactors = FALSE))
        }
    }

    if (nrow(close_data) > 0) {
        close_long <- melt(close_data, id.vars = "method",
                           variable.name = "metric", value.name = "rate")
        close_long$metric <- ifelse(close_long$metric == "close_rate",
                                    ">=95% of DIIM", "Beats DIIM")

        p2 <- ggplot(close_long, aes(x = reorder(method, rate), y = rate, fill = metric)) +
            geom_bar(stat = "identity", position = "dodge", alpha = 0.85) +
            coord_flip() +
            scale_y_continuous(labels = scales::percent) +
            scale_fill_manual(values = c(">=95% of DIIM" = "#3498DB",
                                         "Beats DIIM" = "#2ECC71")) +
            labs(title = sprintf("%s: How Often Each Cheap Method Matches DIIM", scenario_name),
                 x = "", y = "Rate", fill = "") +
            theme_minimal() +
            theme(plot.title = element_text(face = "bold", size = 14),
                  legend.position = "bottom")
        ggsave(file.path(results_dir, paste0(prefix, "_close_rates.png")),
               p2, width = 10, height = 6)
        cat(sprintf("  Saved %s_close_rates.png\n", prefix))
    }

    # --- Plot 3: Decision boundary for each method with a rule ---
    for (rule in rules) {
        if (is.null(rule$best_predictor) || is.na(rule$best_predictor)) next
        if (is.na(rule$threshold)) next

        prefix_m  <- method_prefixes[rule$method]
        close_col <- paste0(prefix_m, "_close")
        pred_col  <- rule$best_predictor

        if (!pred_col %in% names(mc_df) || !close_col %in% names(mc_df)) next

        p3 <- ggplot(mc_df, aes_string(x = pred_col, y = paste0(prefix_m, "_ratio"))) +
            geom_point(aes_string(color = close_col), alpha = 0.3, size = 1) +
            geom_vline(xintercept = rule$threshold, linetype = "dashed",
                       color = "red", linewidth = 1) +
            geom_hline(yintercept = 0.95, linetype = "dotted", color = "blue") +
            scale_color_manual(
                values = c("TRUE" = "#2ECC71", "FALSE" = "#E74C3C"),
                labels = c("TRUE" = ">=95%", "FALSE" = "<95%"),
                name = "Close to DIIM") +
            labs(title = sprintf("%s: %s Decision Boundary", scenario_name, rule$method),
                 subtitle = sprintf("Rule: Use %s if %s %s %.3f",
                     rule$method, pred_col, rule$direction, rule$threshold),
                 x = pred_col, y = "Performance Ratio") +
            theme_minimal() +
            theme(plot.title = element_text(face = "bold", size = 13))

        fname <- paste0(prefix, "_boundary_", tolower(prefix_m), ".png")
        ggsave(file.path(results_dir, fname), p3, width = 10, height = 7)
        cat(sprintf("  Saved %s\n", fname))
    }
}

## 7. Run COVID-19 Scenario

In [ ]:
covid_mc <- run_monte_carlo("COVID-19", download_data,
    lockdown_duration = 55, total_duration = 751,
    days_in_year = 366, n_mc = 2000, k = 5)

covid_rules <- build_decision_rules(covid_mc$mc_df, "COVID-19", method_names)

cat("\nCOVID-19 Plots:\n")
generate_plots(covid_mc$mc_df, covid_rules, "COVID-19", "covid_dr")

## 8. Run Manpower Scenario

In [ ]:
manpower_mc <- run_monte_carlo("Manpower", download_manpower_data,
    lockdown_duration = 55, total_duration = 751,
    days_in_year = 365, n_mc = 2000, k = 5)

manpower_rules <- build_decision_rules(manpower_mc$mc_df, "Manpower", method_names)

cat("\nManpower Plots:\n")
generate_plots(manpower_mc$mc_df, manpower_rules, "Manpower", "manpower_dr")

## 9. Save Combined Data & Summary

In [ ]:
# save combined MC data
combined_mc <- bind_rows(covid_mc$mc_df, manpower_mc$mc_df)
write.csv(combined_mc, file.path(results_dir, "decision_rules_mc_data.csv"), row.names = FALSE)
cat("Saved decision_rules_mc_data.csv\n\n")

# print summary
cat("DECISION RULES SUMMARY\n\n")
summary_lines <- character()

for (sc_name in c("COVID-19", "Manpower")) {
    rules <- if (sc_name == "COVID-19") covid_rules else manpower_rules
    cat(sprintf("--- %s ---\n", sc_name))
    summary_lines <- c(summary_lines, sprintf("--- %s ---", sc_name))

    for (m in names(rules)) {
        r <- rules[[m]]
        rule_str <- if (!is.na(r$threshold) && !is.null(r$best_predictor) && !is.na(r$best_predictor)) {
            sprintf("Use if %s %s %.3f (J=%.3f)",
                    r$best_predictor, r$direction, r$threshold, r$youden_j)
        } else {
            "No clear decision boundary"
        }

        line <- sprintf("  %-25s close=%.1f%% wins=%.1f%% ratio=%.3f | %s",
            m, r$close_rate * 100, r$win_rate * 100, r$mean_ratio, rule_str)
        cat(line, "\n")
        summary_lines <- c(summary_lines, line)
    }
    cat("\n")
    summary_lines <- c(summary_lines, "")
}

# save summary to text file
writeLines(summary_lines, file.path(results_dir, "decision_rules_summary.txt"))
cat("Saved decision_rules_summary.txt\n")
cat("Done!\n")